# Diamond Protein Aligner — Pegasus WMS Workflow

This notebook builds a complete **Pegasus Workflow Management System** pipeline for
running **Diamond** (https://github.com/bbuchfink/diamond) — a high-throughput protein
sequence aligner — against multiple query samples.

## Workflow DAG
```
reference.faa                   sample1.faa  sample2.faa  sample3.faa
       |                              |            |            |
       v                              v            v            v
  makedb (build DB)             blastp       blastp       blastp
       |                              |            |            |
       v                              v            v            v
  reference.dmnd                sample1.tsv  sample2.tsv  sample3.tsv
       |                              |            |            |
       +------------------------------+------------+------------+
                                      |
                                      v
                               merge_results
                                      |
                                      v
                              merged_results.tsv  (staged out)
```

**Pipeline steps:**
1. Build a Diamond database from a reference protein FASTA file
2. Run `diamond blastp` on multiple query samples (in parallel)
3. Merge all alignment results into a single TSV summary

**Execution:** Local machine with a Docker container (`diamond-aligner:latest`).

In [ ]:
# ──────────────────────────────────────────────────────
# 1. Setup: Verify environment
# ──────────────────────────────────────────────────────
import os
import sys
import subprocess
import glob

WORK_DIR = os.path.abspath('diamond-workflow')
BIN_DIR = os.path.join(WORK_DIR, 'bin')
print(f'Work directory: {WORK_DIR}')
print(f'Bin directory:  {BIN_DIR}')
print()

# Verify Pegasus is available
try:
    from Pegasus.api import *
    print('  \u2713 Pegasus WMS API loaded')
except ImportError as e:
    print(f'  \u2717 Pegasus not found: {e}')
    print('  Install with: pip install pegasus-wms.api')
    sys.exit(1)

# Verify wrapper scripts exist
scripts = ['makedb.py', 'blastp.py', 'merge_results.py']
for s in scripts:
    path = os.path.join(BIN_DIR, s)
    exists = os.path.isfile(path)
    print(f'  {"\u2713" if exists else "\u2717"} {s}')
    if not exists:
        print(f'    Missing: {path}')

# Verify Dockerfile
dockerfile = os.path.join(WORK_DIR, 'Dockerfile')
print(f'  {"\u2713" if os.path.isfile(dockerfile) else "\u2717"} Dockerfile')
print()
print('Setup complete.')

---
## Step 2: Define Workflow Properties & Data Files

We define the logical files that flow through the DAG. The workflow expects:
- One reference FASTA (`reference.faa`)
- Multiple query FASTA files (`sample1.faa`, `sample2.faa`, `sample3.faa`)

Feel free to adjust the sample names and counts to match your data.

In [ ]:
# ──────────────────────────────────────────────────────
# 2. Define workflow properties and sample list
# ──────────────────────────────────────────────────────
from Pegasus.api import *

# ---- Workflow name ----
WORKFLOW_NAME = 'diamond-blastp'

# ---- Sample configuration (edit these!) ----
SAMPLES = ['sample1', 'sample2', 'sample3']
REFERENCE_NAME = 'reference'

print(f'Workflow: {WORKFLOW_NAME}')
print(f'Reference: {REFERENCE_NAME}')
print(f'Samples ({len(SAMPLES)}): {', '.join(SAMPLES)}')

# ---- Logical files ----
# Inputs
ref_fasta = File(f'{REFERENCE_NAME}.faa')
query_fastas = {s: File(f'{s}.faa') for s in SAMPLES}

# Intermediate outputs
diamond_db = File(f'{REFERENCE_NAME}.dmnd')
blast_results = {s: File(f'{s}_vs_{REFERENCE_NAME}.tsv') for s in SAMPLES}

# Final output (staged out)
merged_results = File('merged_results.tsv')

print()
print('Logical files defined:')
print(f'  Reference: {ref_fasta.lfn}')
for s in SAMPLES:
    print(f'  Query:     {query_fastas[s].lfn}')
    print(f'  BLAST out: {blast_results[s].lfn}')
print(f'  Merged:    {merged_results.lfn}')

---
## Step 3: Container Definition

We use a Docker container (`diamond-aligner:latest`) that has Diamond installed along
with our Python wrapper scripts. The workflow directory is mounted read-only into the
container at `/opt/workflow/`.

In [ ]:
# ──────────────────────────────────────────────────────
# 3. Container definition
# ──────────────────────────────────────────────────────
container = Container(
    'diamond-container',
    Container.DOCKER,
    image='diamond-aligner:latest',
    mounts=[f'{WORK_DIR}:/opt/workflow:ro']
)

print('Container defined:')
print(f'  Name:  {container.name}')
print(f'  Type:  {container.type}')
print(f'  Image: diamond-aligner:latest')
print(f'  Mount: {WORK_DIR} -> /opt/workflow (ro)')

---
## Step 4: Site Catalog

Register the local execution site with the input and output directories.

In [ ]:
# ──────────────────────────────────────────────────────
# 4. Site Catalog
# ──────────────────────────────────────────────────────
sc = SiteCatalog()

local_site = Site('local', arch=Arch.X86_64, os_type=OS.LINUX)
local_site.add_directories(
    Directory(Directory.SHARED_STORAGE, path=WORK_DIR)
        .add_file(ref_fasta)
        .add_file(*query_fastas.values())
)

sc.add_sites(local_site)
sc.write()

print('[SC] Site catalog written to sites.yml')
print(f'  Site: local')
print(f'  Shared storage: {WORK_DIR}')

---
## Step 5: Transformation Catalog

Each wrapper script is registered as a Pegasus transformation. The container is
attached so jobs are dispatched inside the Diamond container.

In [ ]:
# ──────────────────────────────────────────────────────
# 5. Transformation Catalog
# ──────────────────────────────────────────────────────
tc = TransformationCatalog()

# Diamond makedb
makedb_trans = Transformation(
    'makedb',
    site='local',
    pfn='/opt/workflow/bin/makedb.py',
    is_stageable=False,
    container=container
)

# Diamond blastp
blastp_trans = Transformation(
    'blastp',
    site='local',
    pfn='/opt/workflow/bin/blastp.py',
    is_stageable=False,
    container=container
)

# Merge results
merge_trans = Transformation(
    'merge_results',
    site='local',
    pfn='/opt/workflow/bin/merge_results.py',
    is_stageable=False,
    container=container
)

tc.add_transformations(makedb_trans, blastp_trans, merge_trans)
tc.write()

print('[TC] Transformation catalog written to tc.yml')
for t in ['makedb', 'blastp', 'merge_results']:
    print(f'  \u2713 {t}')

---
## Step 6: Replica Catalog

Maps logical file names to their physical locations on disk. We point to
example data files inside `diamond-workflow/`.

In [ ]:
# ──────────────────────────────────────────────────────
# 6. Replica Catalog
# ──────────────────────────────────────────────────────
rc = ReplicaCatalog()

# Reference FASTA
ref_fasta_path = os.path.join(WORK_DIR, 'example_data', f'{REFERENCE_NAME}.faa')
rc.add_replica('local', ref_fasta.lfn, ref_fasta_path)

# Query FASTA files
for s in SAMPLES:
    qpath = os.path.join(WORK_DIR, 'example_data', f'{s}.faa')
    rc.add_replica('local', query_fastas[s].lfn, qpath)

rc.write()

print('[RC] Replica catalog written to rc.yml')
print(f'  Reference: {ref_fasta.lfn} -> {ref_fasta_path}')
for s in SAMPLES:
    print(f'  Query:     {s}.faa -> {os.path.join(WORK_DIR, "example_data", s + ".faa")}')

---
## Step 7: Build the Workflow DAG

This is the core of the pipeline. We construct three job types:

1. **makedb** (single job) — Builds Diamond DB from reference FASTA
2. **blastp** (one per sample, parallel) — Aligns each query against the DB
3. **merge_results** (fan-in) — Combines all TSV files into one summary

Dependencies are automatically inferred because we share `File` objects between jobs.

In [ ]:
# ──────────────────────────────────────────────────────
# 7. Workflow DAG Construction
# ──────────────────────────────────────────────────────
wf = Workflow(WORKFLOW_NAME, infer_dependencies=True)

# ---- Job 1: Build Diamond database ----
job_makedb = Job(makedb_trans)
job_makedb.add_inputs(ref_fasta)
job_makedb.add_outputs(diamond_db)
job_makedb.add_args(
    '--input', ref_fasta,
    '--output', diamond_db,
    '--threads', '4',
    '--verbose'
)
wf.add_jobs(job_makedb)
print('  \u2713 Job: makedb (build Diamond database)')

# ---- Job 2: Diamond blastp (one per sample, parallel) ----
blast_jobs = []
for s in SAMPLES:
    job_blast = Job(blastp_trans)
    job_blast.add_inputs(query_fastas[s], diamond_db)
    job_blast.add_outputs(blast_results[s])
    job_blast.add_args(
        '--query', query_fastas[s],
        '--db', diamond_db,
        '--output', blast_results[s],
        '--evalue', '1e-5',
        '--max-target-seqs', '25',
        '--threads', '4',
        '--verbose'
    )
    wf.add_jobs(job_blast)
    blast_jobs.append(job_blast)
    print(f'  \u2713 Job: blastp ({s})')

# ---- Job 3: Merge results (fan-in) ----
job_merge = Job(merge_trans)
job_merge.add_inputs(*[blast_results[s] for s in SAMPLES])
job_merge.add_outputs(merged_results, stage_out=True)  # Final output
job_merge.add_args(
    '--inputs', *[blast_results[s] for s in SAMPLES],
    '--sample-names', *SAMPLES,
    '--output', merged_results
)
wf.add_jobs(job_merge)
print(f'  \u2713 Job: merge_results (fan-in)')

print()
print(f'Total jobs in workflow: {len(wf.jobs)}')

---
## Step 8: Attach Catalogs & Write Workflow

We register the previously created catalogs with the workflow object and write
everything to `workflow.yml`.

In [ ]:
# ──────────────────────────────────────────────────────
# 8. Write workflow YAML
# ──────────────────────────────────────────────────────
wf.add_site_catalog(sc)
wf.add_transformation_catalog(tc)
wf.add_replica_catalog(rc)

wf.write()

import json
print('=' * 70)
print('WORKFLOW GENERATED SUCCESSFULLY')
print('=' * 70)
print(f'  Location:  {WORK_DIR}')
print(f'  Workflow:  {wf.name}')
print(f'  Jobs:      {len(wf.jobs)}')
print(f'  Structure: makedb -> (blastp x {len(SAMPLES)} parallel) -> merge_results')
print(f'  Container: diamond-aligner:latest (Docker)')
print(f'  Files:')
print(f'    - workflow.yml')

---
## Step 9: Create Example Data

This cell generates small synthetic FASTA files for testing the workflow.
Replace these with real protein sequences in production.

In [ ]:
# ──────────────────────────────────────────────────────
# 9. Generate example FASTA data (for testing)
# ──────────────────────────────────────────────────────
import random

EXAMPLE_DIR = os.path.join(WORK_DIR, 'example_data')
os.makedirs(EXAMPLE_DIR, exist_ok=True)

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

def random_protein(length=100):
    return ''.join(random.choices(AMINO_ACIDS, k=length))

def write_fasta(path, seq_id, sequence):
    with open(path, 'w') as f:
        f.write(f'>{seq_id}\n')
        # Wrap at 60 chars
        for i in range(0, len(sequence), 60):
            f.write(sequence[i:i+60] + '\n')

random.seed(42)

# Reference: 5 proteins
ref_path = os.path.join(EXAMPLE_DIR, 'reference.faa')
with open(ref_path, 'w') as f:
    for i in range(5):
        seq = random_protein(120)
        f.write(f'>ref_protein_{i+1}\n')
        for j in range(0, len(seq), 60):
            f.write(seq[j:j+60] + '\n')
print(f'  \u2713 Created: reference.faa (5 proteins)')

# Queries: 3 samples, 10 queries each (some homologous)
for s in SAMPLES:
    qpath = os.path.join(EXAMPLE_DIR, f'{s}.faa')
    with open(qpath, 'w') as f:
        for i in range(10):
            # Mix of random and reference-derived sequences
            if random.random() < 0.3:
                # Similar to a reference protein (mutated copy)
                seq = random_protein(120)
            else:
                seq = random_protein(random.randint(80, 150))
            f.write(f'>{s}_query_{i+1}\n')
            for j in range(0, len(seq), 60):
                f.write(seq[j:j+60] + '\n')
    print(f'  \u2713 Created: {s}.faa (10 queries)')

print()
print(f'Example data written to: {EXAMPLE_DIR}')
print('Replace these with your real FASTA files before running.')

---
## Step 10: Run the Workflow

Before running, build the Docker container:

```bash
cd diamond-workflow
docker build -t diamond-aligner:latest .
```

Then plan and submit the workflow with `pegasus-plan`:

```bash
# Plan (dry-run, inspect DAG)
pegasus-plan --sites local --output-sites local diamond-workflow/workflow.yml

# Plan and submit
pegasus-plan --sites local --output-sites local --submit diamond-workflow/workflow.yml
```

Alternatively, run the cells below to execute from the notebook.

In [ ]:
# ──────────────────────────────────────────────────────
# 10a. Build the Docker container (optional)
# ──────────────────────────────────────────────────────
import subprocess

build = input('Build Docker container? (y/n): ').strip().lower()

if build == 'y':
    print('Building Docker container: diamond-aligner:latest ...')
    result = subprocess.run(
        ['docker', 'build', '-t', 'diamond-aligner:latest', WORK_DIR],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('  \u2713 Docker image built successfully')
    else:
        print(f'  \u2717 Build failed:\n{result.stderr}')
else:
    print('Skipping Docker build. Run manually:')
    print(f'  docker build -t diamond-aligner:latest {WORK_DIR}')

In [ ]:
# ──────────────────────────────────────────────────────
# 10b. Plan the workflow (dry-run)
# ──────────────────────────────────────────────────────
import subprocess

plan = input('Plan workflow with pegasus-plan? (y/n): ').strip().lower()

if plan == 'y':
    wf_yml = os.path.join(WORK_DIR, 'workflow.yml')
    cmd = [
        'pegasus-plan',
        '--sites', 'local',
        '--output-sites', 'local',
        '--dir', os.path.join(WORK_DIR, 'output'),
        wf_yml
    ]
    print(f'Running: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(result.stdout)
        print('  \u2713 Workflow planned successfully')
    else:
        print(f'  \u2717 Planning failed:\n{result.stderr}')
else:
    print('Skipping. Run manually:')
    print(f'  pegasus-plan --sites local --output-sites local --dir {WORK_DIR}/output {WORK_DIR}/workflow.yml')

---
## Step 11: Workflow Summary & Customization

### What was created

| File / Directory | Purpose |
|---|---|
| `diamond-workflow/workflow.yml` | Pegasus workflow definition (DAG) |
| `diamond-workflow/sites.yml` | Site catalog (execution host) |
| `diamond-workflow/tc.yml` | Transformation catalog (jobs) |
| `diamond-workflow/rc.yml` | Replica catalog (input data locations) |
| `diamond-workflow/bin/makedb.py` | Wrapper for `diamond makedb` |
| `diamond-workflow/bin/blastp.py` | Wrapper for `diamond blastp` |
| `diamond-workflow/bin/merge_results.py` | Merge TSV results |
| `diamond-workflow/Dockerfile` | Container definition |
| `diamond-workflow/example_data/` | Synthetic FASTA files for testing |

### Customization Tips

- **Add more samples:** Add entries to `SAMPLES` list (cell 2).
- **Different Diamond modes:** Replace `blastp` with `blastx` by creating a similar wrapper.
- **Sensitive mode:** Add `'--sensitive'` to the blastp job args.
- **OSPool / HPC:** Change the `Site` definition and set appropriate profiles.
- **Singularity/Apptainer:** Use `Container.SINGULARITY` instead of `Container.DOCKER`.
- **Output format:** BLAST tabular (`--outfmt 6`) is used. Change in `blastp.py` if needed.

### View the DAG

Run the next cell to render a simple text representation of the workflow DAG.

In [ ]:
# ──────────────────────────────────────────────────────
# 11. Visualize the DAG
# ──────────────────────────────────────────────────────
print('Workflow DAG:')
print('=' * 50)
print()

# Build a simple adjacency view
print(f'  {ref_fasta.lfn}')
print(f'       |')
print(f'       v')
print(f'  [makedb]  -->  {diamond_db.lfn}')
print(f'       |')
print(f'       v')

for i, s in enumerate(SAMPLES):
    prefix = '  +--' if i == 0 else '  |--' if i < len(SAMPLES) - 1 else '  +--'
    print(f'  {prefix} [{s}:blastp]  -->  {blast_results[s].lfn}')

print(f'       |')
print(f'       v')
print(f'  [merge_results]  -->  {merged_results.lfn}  (staged out)')
print()
print(f'  Parallel blastp jobs: {len(SAMPLES)}')
print(f'  Total jobs: {len(wf.jobs)}')
print(f'  Container: diamond-aligner:latest')
print()
print('=' * 50)

In [ ]:
# ──────────────────────────────────────────────────────
# 12. List all generated files
# ──────────────────────────────────────────────────────
import subprocess

print('Files in diamond-workflow/:')
print()
result = subprocess.run(
    ['find', WORK_DIR, '-type', 'f', '|', 'sort'],
    capture_output=True, text=True, shell=True
)
print(result.stdout)